# 🤖 PatrolIQ — ML Training Pipeline

### What this notebook does
This notebook implements the **complete unsupervised machine learning pipeline** for the PatrolIQ platform.

We apply **5 algorithms** on 505K Chicago crime records and track every experiment using **MLflow**.

---
### 📌 Table of Contents
1. [Setup & Imports](#1-setup)
2. [Load Cleaned Data](#2-load)
3. [Geographic Clustering — K-Means](#3-kmeans)
4. [Geographic Clustering — DBSCAN](#4-dbscan)
5. [Geographic Clustering — Hierarchical](#5-hierarchical)
6. [Temporal Clustering — K-Means](#6-temporal)
7. [Dimensionality Reduction — PCA](#7-pca)
8. [Dimensionality Reduction — t-SNE](#8-tsne)
9. [Model Comparison & Final Report](#9-compare)

---
### 🔑 Key Concepts
| Term | Meaning |
|------|---------|
| **Unsupervised Learning** | No labels/targets — model discovers natural groups |
| **Clustering** | Group similar data points together |
| **Silhouette Score** | Measures how well-separated clusters are (0–1, higher = better) |
| **Davies-Bouldin Index** | Measures cluster overlap (lower = better) |
| **MLflow** | Experiment tracking platform — logs params, metrics, models |

## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

# Clustering algorithms from scikit-learn
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering

# Dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Evaluation metrics
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Hierarchical dendrogram
from scipy.cluster.hierarchy import dendrogram, linkage

# MLflow experiment tracking
import mlflow
import mlflow.sklearn

warnings.filterwarnings('ignore')

# ── File Paths ──────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
CLEAN_DIR    = os.path.join(PROJECT_ROOT, 'data', 'cleaned')
MLFLOW_URI   = os.path.join(PROJECT_ROOT, 'mlruns')
EXPERIMENT   = 'PatrolIQ_Crime_Analysis'

# Sample sizes — we use subsets for speed
GEO_SAMPLE = 50_000   # 50K records for geographic clustering
ML_SAMPLE  = 15_000   # 15K records for PCA / t-SNE (computationally expensive)

# Setup MLflow
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)

print('✅ All imports successful')
print(f'📂 Data directory : {CLEAN_DIR}')
print(f'📂 MLflow URI     : {MLFLOW_URI}')

## 2. Load Cleaned Data <a id='2-load'></a>

In [ ]:
# Load the cleaned CSV produced by the preprocessing notebook
clean_path = os.path.join(CLEAN_DIR, 'cleaned_crimes.csv')
print(f'Loading {clean_path}...')

df = pd.read_csv(clean_path, low_memory=False)
print(f'\n✅ Loaded! Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Quick view of the features we'll use
feature_cols = ['Hour', 'Day_Num', 'Month', 'Is_Weekend', 'Is_Night',
                'Crime_Severity_Score', 'Latitude', 'Longitude',
                'District', 'Ward', 'Beat', 'Community Area', 'Arrest', 'Domestic']

print(f'\nML Feature columns: {feature_cols}')
df[feature_cols].describe().round(3)

---
## 3. Geographic Clustering — K-Means <a id='3-kmeans'></a>

### 💡 What is K-Means?
K-Means partitions data into **k pre-defined clusters** by iteratively:
1. Assigning each point to its nearest **centroid** (cluster center)
2. Recalculating centroids as the **mean** of all assigned points
3. Repeating until convergence (centroids stop moving)

### Why use it for crime data?
- Creates **k fixed patrol zones** (e.g., 8 police districts)
- Fast and interpretable — cluster centers = ideal patrol checkpoint locations
- We use **Latitude/Longitude only** so clusters are purely geographic

### StandardScaler — Why scale the data?
Latitude values (~41.8) and Longitude values (~-87.7) are on different scales to each other.  
StandardScaler transforms each feature to **mean=0, std=1** so neither dominates the distance calculation.

In [ ]:
# ── Prepare Geographic Sample ────────────────────────────────────────────────
# We use 5 columns: Lat/Lon for clustering, Arrest/Primary Type/Severity for analysis
geo = df[['Latitude', 'Longitude', 'Arrest', 'Primary Type', 'Crime_Severity_Score']].dropna()

# Sample 50K points (faster than 505K, statistically representative)
# random_state=42 ensures reproducibility (same result every run)
geo_sample = geo.sample(n=min(GEO_SAMPLE, len(geo)), random_state=42).reset_index(drop=True)

print(f'Geographic sample: {geo_sample.shape}')
print(f'Lat range: {geo_sample["Latitude"].min():.4f} — {geo_sample["Latitude"].max():.4f}')
print(f'Lon range: {geo_sample["Longitude"].min():.4f} — {geo_sample["Longitude"].max():.4f}')

# ── Scale Features ───────────────────────────────────────────────────────────
# K-Means uses Euclidean distance — features MUST be on the same scale
X = geo_sample[['Latitude', 'Longitude']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nAfter scaling:')
print(f'  Mean: {X_scaled.mean(axis=0).round(6)}  (should be ≈ 0)')
print(f'  Std : {X_scaled.std(axis=0).round(6)}  (should be ≈ 1)')

In [ ]:
# ── K-Means Clustering ────────────────────────────────────────────────────────
k = 8  # 8 clusters ≈ 8 Chicago police districts

# n_init=10: run K-Means 10 times with different starting centroids, keep best result
# This avoids getting stuck in bad local optima
km = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_km = km.fit_predict(X_scaled)  # Returns cluster ID (0 to k-1) for each point

# ── Evaluation Metrics ───────────────────────────────────────────────────────
# Silhouette Score: measures how similar a point is to its own cluster vs other clusters
# Range: -1 (bad) to +1 (perfect). Above 0.5 = good separation
# sample_size=5000: compute on subset for speed (exact on full data would be very slow)
sil_km = silhouette_score(X_scaled, labels_km, sample_size=5000, random_state=42)

# Davies-Bouldin: ratio of within-cluster scatter to between-cluster separation
# Lower is better. 0 = perfect clusters, >2 = poor separation
db_km = davies_bouldin_score(X_scaled, labels_km)

print(f'K-Means Results:')
print(f'  Silhouette Score  : {sil_km:.4f}  (higher = better, max 1.0)')
print(f'  Davies-Bouldin    : {db_km:.4f}  (lower = better, min 0.0)')
print(f'  Cluster sizes     : {pd.Series(labels_km).value_counts().sort_index().to_dict()}')

# Decode cluster centers back to real Lat/Lon coordinates
centers = pd.DataFrame(scaler.inverse_transform(km.cluster_centers_),
                       columns=['Latitude', 'Longitude'])
print(f'\nCluster Centers (real coordinates):')
print(centers.round(4).to_string())

In [ ]:
# ── Log to MLflow ──────────────────────────────────────────────────────────────
# MLflow records EVERY experiment run so you can compare them later
with mlflow.start_run(run_name='KMeans_Geographic'):
    # Log hyperparameters
    mlflow.log_params({'algorithm': 'KMeans', 'n_clusters': k, 'type': 'geographic'})
    # Log evaluation metrics
    mlflow.log_metrics({'silhouette_score': round(sil_km, 4), 'davies_bouldin': round(db_km, 4)})
    # Log the trained model (can be loaded later for predictions)
    mlflow.sklearn.log_model(km, 'kmeans_geo_model')
    print('✅ K-Means run logged to MLflow')

# ── Save Results ───────────────────────────────────────────────────────────────
geo_sample['KMeans_Geo'] = labels_km
centers.to_csv(os.path.join(CLEAN_DIR, 'kmeans_geo_centers.csv'), index=False)

# Store metrics for later comparison
geo_metrics = {
    'KMeans_Geo': {'silhouette': round(sil_km,4), 'davies_bouldin': round(db_km,4), 'n_clusters': k}
}

# ── Visualisation ──────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 7))
colors = plt.cm.tab10(np.linspace(0, 1, k))
for i in range(k):
    mask = labels_km == i
    plt.scatter(geo_sample.loc[mask, 'Longitude'], geo_sample.loc[mask, 'Latitude'],
                s=2, alpha=0.4, color=colors[i], label=f'Cluster {i}')

# Plot centroids as stars
plt.scatter(centers['Longitude'], centers['Latitude'],
            marker='*', s=300, c='black', zorder=5, label='Centroids')

plt.title(f'K-Means Geographic Clustering (k={k})\nSilhouette={sil_km:.4f}, DB={db_km:.4f}',
          fontsize=13, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=5)
plt.tight_layout()
plt.show()

---
## 4. Geographic Clustering — DBSCAN <a id='4-dbscan'></a>

### 💡 What is DBSCAN?
**Density-Based Spatial Clustering of Applications with Noise**

DBSCAN works by finding **dense regions** of points:
- A point is a **core point** if it has ≥ `min_samples` neighbours within radius `eps`
- **Border points** are near a core point but don't have enough neighbours themselves
- **Noise points** (label = -1) are isolated — too far from any dense region

### Key advantage over K-Means:
- ✅ Does **not** require specifying k upfront
- ✅ Discovers arbitrarily shaped clusters (not just circles)
- ✅ Naturally handles **outliers** (noise points = isolated crimes)
- ❌ Sensitive to eps and min_samples parameters

### Parameter tuning:
- `eps=0.15` — in **scaled** space, 0.15 ≈ ~1.5km radius in Chicago
- `min_samples=10` — a crime hotspot needs at least 10 incidents to be a 'cluster'

In [ ]:
# ── DBSCAN Clustering ────────────────────────────────────────────────────────
eps, min_s = 0.15, 10

db_model = DBSCAN(eps=eps, min_samples=min_s)
labels_db = db_model.fit_predict(X_scaled)
# Label -1 = noise (points that don't belong to any cluster)

# Count clusters (exclude -1 noise label)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
noise_count   = (labels_db == -1).sum()
noise_pct     = round(100 * noise_count / len(labels_db), 1)

print(f'DBSCAN Results:')
print(f'  Clusters found   : {n_clusters_db}')
print(f'  Noise points     : {noise_count:,} ({noise_pct}% of data)')

# ── Silhouette (only on non-noise points, only if >1 cluster) ────────────────
# Silhouette score requires at least 2 clusters and can't include noise points
if n_clusters_db > 1:
    mask_no_noise = labels_db != -1
    sil_db  = silhouette_score(X_scaled[mask_no_noise], labels_db[mask_no_noise],
                               sample_size=5000, random_state=42)
    db_score = davies_bouldin_score(X_scaled[mask_no_noise], labels_db[mask_no_noise])
else:
    sil_db, db_score = 0.0, 0.0

print(f'  Silhouette Score : {sil_db:.4f}')
print(f'  Davies-Bouldin   : {db_score:.4f}')
print(f'  Cluster sizes    :')
cluster_sizes = pd.Series(labels_db).value_counts().sort_index()
print(cluster_sizes.to_string())

In [ ]:
# ── Log to MLflow ──────────────────────────────────────────────────────────────
with mlflow.start_run(run_name='DBSCAN_Geographic'):
    mlflow.log_params({'algorithm': 'DBSCAN', 'eps': eps, 'min_samples': min_s, 'type': 'geographic'})
    mlflow.log_metrics({'silhouette_score': round(sil_db, 4), 'davies_bouldin': round(db_score, 4),
                        'n_clusters': n_clusters_db, 'noise_pct': noise_pct})
    print('✅ DBSCAN run logged to MLflow')

geo_sample['DBSCAN_Geo'] = labels_db
geo_metrics['DBSCAN_Geo'] = {'silhouette': round(sil_db,4), 'davies_bouldin': round(db_score,4),
                              'n_clusters': n_clusters_db, 'noise_pct': noise_pct}

# ── Visualisation ──────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 7))
unique_labels = sorted(set(labels_db))
cmap = plt.cm.tab10

for label in unique_labels:
    mask = labels_db == label
    if label == -1:
        plt.scatter(geo_sample.loc[mask, 'Longitude'], geo_sample.loc[mask, 'Latitude'],
                    s=1, alpha=0.2, color='#cccccc', label='Noise')
    else:
        color = cmap(label % 10)
        plt.scatter(geo_sample.loc[mask, 'Longitude'], geo_sample.loc[mask, 'Latitude'],
                    s=3, alpha=0.5, color=color, label=f'Cluster {label}')

plt.title(f'DBSCAN Geographic Clustering (eps={eps}, min_samples={min_s})\n'
          f'Clusters={n_clusters_db}, Noise={noise_pct}%, Silhouette={sil_db:.4f}',
          fontsize=12, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

---
## 5. Geographic Clustering — Hierarchical (Agglomerative) <a id='5-hierarchical'></a>

### 💡 What is Hierarchical Clustering?
Agglomerative clustering builds clusters **bottom-up**:
1. Start: each point is its own cluster (N clusters)
2. Repeatedly **merge** the two most similar clusters
3. Stop when you reach k clusters

The result can be visualised as a **dendrogram** — a tree showing how clusters were merged.

### Ward Linkage:
We use `linkage='ward'` which merges clusters to **minimise the increase in total within-cluster variance**.  
This tends to produce compact, equally-sized clusters — ideal for patrol zone planning.

### Why is it slower?
Agglomerative clustering requires computing pairwise distances → **O(n²) memory**.  
That's why we still use the full 50K sample for labeling, but only 500 points for the dendrogram.

In [ ]:
# ── Hierarchical Clustering ───────────────────────────────────────────────────
k_hier = 8

# Fit on ALL 50K points to get cluster labels
hier_full = AgglomerativeClustering(n_clusters=k_hier, linkage='ward')
labels_hier = hier_full.fit_predict(X_scaled)

# Evaluate
sil_h = silhouette_score(X_scaled, labels_hier, sample_size=5000, random_state=42)
db_h  = davies_bouldin_score(X_scaled, labels_hier)

print(f'Hierarchical Clustering Results:')
print(f'  Clusters         : {k_hier}')
print(f'  Silhouette Score : {sil_h:.4f}')
print(f'  Davies-Bouldin   : {db_h:.4f}')

# ── Log to MLflow ──────────────────────────────────────────────────────────────
with mlflow.start_run(run_name='Hierarchical_Geographic'):
    mlflow.log_params({'algorithm': 'Hierarchical', 'n_clusters': k_hier, 'linkage': 'ward', 'type': 'geographic'})
    mlflow.log_metrics({'silhouette_score': round(sil_h, 4), 'davies_bouldin': round(db_h, 4)})
    print('✅ Hierarchical run logged to MLflow')

geo_sample['Hierarchical_Geo'] = labels_hier
geo_metrics['Hierarchical_Geo'] = {'silhouette': round(sil_h,4), 'davies_bouldin': round(db_h,4), 'n_clusters': k_hier}

In [ ]:
# ── Dendrogram Visualisation ──────────────────────────────────────────────────
# A dendrogram shows the hierarchy of cluster merges
# We use only 500 points (scipy's linkage is O(n^2) in memory)
# truncate_mode='lastp', p=30: only show the last 30 merge steps (keeps it readable)

Z = linkage(X_scaled[:500], method='ward')

fig, ax = plt.subplots(figsize=(15, 5))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30,
           leaf_font_size=9, color_threshold=None)

ax.set_title('Hierarchical Clustering Dendrogram — Chicago Crime Hotspots\n'
             '(X-axis = samples merged, Y-axis = merge distance/cost)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Sample Index / Cluster Size (in brackets)')
ax.set_ylabel('Ward Linkage Distance')

# Add a horizontal line at the suggested cut point for k=8
ax.axhline(y=Z[-k_hier+1, 2], color='red', linestyle='--',
           label=f'Cut for k={k_hier} clusters')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CLEAN_DIR, 'dendrogram_geo.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Dendrogram saved!')

In [ ]:
# ── Save All Geo Clustering Results ──────────────────────────────────────────
geo_sample.to_csv(os.path.join(CLEAN_DIR, 'geo_sample.csv'), index=False)

with open(os.path.join(CLEAN_DIR, 'geo_clustering_metrics.json'), 'w') as f:
    json.dump(geo_metrics, f, indent=2)

# Per-cluster stats for each algorithm (used in the Streamlit app tables)
for algo in ['KMeans_Geo', 'DBSCAN_Geo', 'Hierarchical_Geo']:
    src = geo_sample[geo_sample[algo] != -1] if algo == 'DBSCAN_Geo' else geo_sample
    stats = src.groupby(algo).agg(
        Total_Crimes=(algo, 'count'),
        Arrest_Rate=('Arrest', 'mean'),
        Top_Crime=('Primary Type', lambda x: x.value_counts().index[0])
    ).reset_index()
    stats['Arrest_Rate'] = (stats['Arrest_Rate'] * 100).round(1)
    stats.to_csv(os.path.join(CLEAN_DIR, f'{algo.lower()}_stats.csv'), index=False)

print('✅ All geographic clustering results saved!')
print(json.dumps(geo_metrics, indent=2))

---
## 6. Temporal Clustering — K-Means <a id='6-temporal'></a>

### 💡 What are we clustering here?
Instead of **where** crimes happen (Lat/Lon), we now cluster **when** they happen using:  
`Hour, Day_Num, Month, Crime_Severity_Score`

This discovers **4 temporal crime profiles**:
- e.g., Cluster 0 = Weekday afternoon thefts
- e.g., Cluster 1 = Friday/Saturday night violent crimes
- e.g., Cluster 2 = Early morning domestic incidents
- e.g., Cluster 3 = Summer daytime narcotics

These profiles help police schedule **shift allocations** at peak times.

In [ ]:
# ── Prepare Temporal Sample ───────────────────────────────────────────────────
temp_cols = ['Hour', 'Day_Num', 'Month', 'Crime_Severity_Score', 'Is_Weekend', 'Arrest', 'Primary Type']
temp = df[temp_cols].dropna()
temp_sample = temp.sample(n=min(GEO_SAMPLE, len(temp)), random_state=42).reset_index(drop=True)

# Features for clustering (numeric only)
temporal_features = ['Hour', 'Day_Num', 'Month', 'Crime_Severity_Score']
X_temp = temp_sample[temporal_features].values

# Scale so Hour (0-23) doesn't dominate Month (1-12)
scaler_t = StandardScaler()
X_temp_scaled = scaler_t.fit_transform(X_temp)

print(f'Temporal sample shape: {temp_sample.shape}')
print(f'Features: {temporal_features}')

In [ ]:
# ── K-Means Temporal Clustering ───────────────────────────────────────────────
k_temp = 4  # 4 temporal crime profiles

km_t = KMeans(n_clusters=k_temp, random_state=42, n_init=10)
labels_t = km_t.fit_predict(X_temp_scaled)

sil_t = silhouette_score(X_temp_scaled, labels_t, sample_size=5000, random_state=42)
db_t  = davies_bouldin_score(X_temp_scaled, labels_t)

print(f'Temporal K-Means Results:')
print(f'  Silhouette Score : {sil_t:.4f}')
print(f'  Davies-Bouldin   : {db_t:.4f}')

temp_sample['Temporal_Cluster'] = labels_t

# ── Cluster Profile Analysis ──────────────────────────────────────────────────
# Decode cluster centers back to original scale to understand what each cluster means
centers_real = scaler_t.inverse_transform(km_t.cluster_centers_)
cluster_profiles = pd.DataFrame(centers_real, columns=temporal_features)
cluster_profiles['Cluster'] = range(k_temp)

print('\nCluster Center Profiles (in original scale):')
print(cluster_profiles.round(1).to_string(index=False))
print('\n(Hour: 0-23, Day_Num: Mon=0 to Sun=6, Month: 1-12, Severity: 1-10)')

In [ ]:
# ── Log to MLflow ──────────────────────────────────────────────────────────────
with mlflow.start_run(run_name='KMeans_Temporal'):
    mlflow.log_params({'algorithm': 'KMeans', 'n_clusters': k_temp, 'type': 'temporal'})
    mlflow.log_metrics({'silhouette_score': round(sil_t, 4), 'davies_bouldin': round(db_t, 4)})
    mlflow.sklearn.log_model(km_t, 'kmeans_temporal_model')
    print('✅ Temporal clustering logged to MLflow')

temporal_metrics = {'KMeans_Temporal': {'silhouette': round(sil_t,4), 'davies_bouldin': round(db_t,4), 'n_clusters': k_temp}}
with open(os.path.join(CLEAN_DIR, 'temporal_clustering_metrics.json'), 'w') as f:
    json.dump(temporal_metrics, f, indent=2)
temp_sample.to_csv(os.path.join(CLEAN_DIR, 'temporal_sample.csv'), index=False)

# ── Visualise Cluster Profiles ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hour distribution per cluster
for cluster_id in range(k_temp):
    subset = temp_sample[temp_sample['Temporal_Cluster'] == cluster_id]
    axes[0].hist(subset['Hour'], bins=24, alpha=0.5, label=f'Cluster {cluster_id}')
axes[0].set_title('Crime Hours by Temporal Cluster', fontweight='bold')
axes[0].set_xlabel('Hour of Day (0=midnight, 12=noon)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Day distribution per cluster
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
cluster_day = temp_sample.groupby(['Temporal_Cluster', 'Day_Num']).size().unstack(fill_value=0)
cluster_day.T.plot(kind='bar', ax=axes[1], colormap='tab10')
axes[1].set_title('Crime Days by Temporal Cluster', fontweight='bold')
axes[1].set_xlabel('Day of Week')
axes[1].set_xticklabels(day_names, rotation=0)
axes[1].legend(title='Cluster')

plt.tight_layout()
plt.show()

---
## 7. Dimensionality Reduction — PCA <a id='7-pca'></a>

### 💡 What is PCA?
**Principal Component Analysis** reduces many features into fewer **principal components** while retaining as much variance (information) as possible.

### Why do we need it?
Our crime dataset has **14 numeric features**. It's impossible to visualise 14 dimensions.  
PCA compresses them into 2D/3D while preserving the most important patterns.

### How PCA works:
1. Center the data (subtract mean)
2. Find **directions of maximum variance** (eigenvectors = principal components)
3. Project data onto these directions

### Scree Plot:
Shows how much variance each component explains.  
The **elbow** suggests how many components are sufficient.

### Feature Loadings:
How much each original feature contributes to each principal component.  
High loading = that feature strongly influences the component's direction.

In [ ]:
# ── Prepare Features for PCA ──────────────────────────────────────────────────
feature_cols_pca = ['Hour', 'Day_Num', 'Month', 'Is_Weekend', 'Is_Night',
                    'Crime_Severity_Score', 'Latitude', 'Longitude',
                    'District', 'Ward', 'Beat', 'Community Area', 'Arrest', 'Domestic']

ml_df = df[feature_cols_pca].dropna()
ml_sample = ml_df.sample(n=min(ML_SAMPLE, len(ml_df)), random_state=42)

# Scale ALL features — PCA is sensitive to feature scale
scaler_pca = StandardScaler()
X_pca_scaled = scaler_pca.fit_transform(ml_sample)

print(f'PCA input: {ml_sample.shape[0]:,} samples × {ml_sample.shape[1]} features')

In [ ]:
# ── Run Full PCA — Find Components Needed for 70% Variance ───────────────────
# We first run PCA with ALL components to find the cumulative variance curve
pca_full = PCA(random_state=42)
pca_full.fit(X_pca_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

# Find how many components reach each threshold
n_70 = int(np.argmax(cumvar >= 0.70)) + 1   # argmax finds first index where condition is True
n_80 = int(np.argmax(cumvar >= 0.80)) + 1

print(f'Components needed for 70% variance: {n_70}')
print(f'Components needed for 80% variance: {n_80}')
print(f'First 2 components explain        : {cumvar[1]*100:.1f}% variance')

# ── Scree Plot ────────────────────────────────────────────────────────────────
n_show = len(feature_cols_pca)
fig, ax = plt.subplots(figsize=(11, 5))

bars = ax.bar(range(1, n_show+1), pca_full.explained_variance_ratio_[:n_show]*100,
              color='steelblue', alpha=0.7, label='Individual variance')
ax2 = ax.twinx()
ax2.plot(range(1, n_show+1), cumvar[:n_show]*100, 'o-', color='darkorange',
         linewidth=2.5, label='Cumulative variance')

# Add threshold lines
ax2.axhline(70, color='green', linestyle='--', alpha=0.7, label='70% threshold')
ax2.axhline(80, color='red',   linestyle='--', alpha=0.7, label='80% threshold')
ax2.axvline(n_70, color='green', linestyle=':', alpha=0.6)
ax2.axvline(n_80, color='red',   linestyle=':', alpha=0.6)

ax.set_xlabel('Principal Component Number')
ax.set_ylabel('Individual Explained Variance (%)', color='steelblue')
ax2.set_ylabel('Cumulative Variance (%)', color='darkorange')
ax.set_title(f'PCA Scree Plot — {n_70} components → 70%, {n_80} components → 80%',
             fontsize=12, fontweight='bold')
ax.set_xticks(range(1, n_show+1))
ax.set_xticklabels([f'PC{i}' for i in range(1, n_show+1)], rotation=45)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
plt.show()

In [ ]:
# ── PCA 2D for Visualisation ──────────────────────────────────────────────────
# Project data onto the first 2 components for 2D scatter plot
pca_2d = PCA(n_components=2, random_state=42)
X_2d   = pca_2d.fit_transform(X_pca_scaled)
var2d  = float((pca_2d.explained_variance_ratio_.sum() * 100).round(2))

# ── Feature Loadings (PC1, PC2, PC3) ─────────────────────────────────────────
# Loadings = correlation between each original feature and each principal component
# Large positive loading → feature increases along that PC axis
# Large negative loading → feature decreases along that PC axis
pca_3d = PCA(n_components=3, random_state=42)
pca_3d.fit(X_pca_scaled)
loadings_df = pd.DataFrame(
    pca_3d.components_.T,
    index=feature_cols_pca,
    columns=['PC1', 'PC2', 'PC3']
)

print(f'2D PCA explains {var2d}% of total variance')
print('\nFeature Loadings (how much each feature contributes to PC1/PC2/PC3):')
print(loadings_df.round(3).to_string())

# Save outputs
scree_data = pd.DataFrame({
    'Component'         : [f'PC{i+1}' for i in range(len(pca_full.explained_variance_ratio_))],
    'Explained_Variance': pca_full.explained_variance_ratio_.round(4),
    'Cumulative_Variance': cumvar.round(4)
})
scree_data.to_csv(os.path.join(CLEAN_DIR, 'pca_variance.csv'), index=False)
loadings_df.to_csv(os.path.join(CLEAN_DIR, 'pca_loadings.csv'))
print('\nScree data and loadings saved!')

In [ ]:
# ── Save PCA Result + Log to MLflow ──────────────────────────────────────────
pca_result = ml_sample.copy().reset_index(drop=True)
pca_result['PC1'] = X_2d[:, 0]
pca_result['PC2'] = X_2d[:, 1]

idx   = df[feature_cols_pca].dropna().index
types = df.loc[idx, 'Primary Type'].reset_index(drop=True)
pca_result['Primary_Type'] = types[pca_result.index].values
pca_result.to_csv(os.path.join(CLEAN_DIR, 'pca_result.csv'), index=False)

with mlflow.start_run(run_name='PCA_DimensionalityReduction'):
    mlflow.log_params({'n_components_2d': 2, 'n_components_70pct': n_70, 'n_features': 14})
    mlflow.log_metrics({'variance_explained_2d': round(var2d, 2), 'n_components_for_70pct': n_70})
    mlflow.sklearn.log_model(pca_2d, 'pca_2d_model')
    print('✅ PCA logged to MLflow')

dim_metrics = {
    'PCA': {'variance_explained_2pc': round(var2d, 2), 'n_components_70pct': n_70,
             'n_components_80pct': n_80, 'n_features': 14}
}

# ── 2D PCA Scatter Plot ───────────────────────────────────────────────────────
top_types = pca_result['Primary_Type'].value_counts().head(6).index
plot_df   = pca_result[pca_result['Primary_Type'].isin(top_types)].sample(3000, random_state=42)

plt.figure(figsize=(10, 7))
for ctype in top_types:
    mask = plot_df['Primary_Type'] == ctype
    plt.scatter(plot_df.loc[mask, 'PC1'], plot_df.loc[mask, 'PC2'],
                s=8, alpha=0.4, label=ctype)

plt.title(f'PCA 2D Projection — Top 6 Crime Types\n({var2d}% variance explained)',
          fontsize=12, fontweight='bold')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(markerscale=4, bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

---
## 8. Dimensionality Reduction — t-SNE <a id='8-tsne'></a>

### 💡 What is t-SNE?
**t-Distributed Stochastic Neighbor Embedding** is a non-linear dimensionality reduction technique ideal for **visualising high-dimensional clusters**.

### How t-SNE differs from PCA:
| | PCA | t-SNE |
|-|-----|-------|
| Type | Linear | Non-linear |
| Speed | Fast | Slow |
| Global structure | Preserves | May distort |
| Local clusters | Partial | **Excellent** |
| Use case | Variance summary | Cluster exploration |

### Key parameters:
- `perplexity=30` — roughly "how many local neighbours to consider" (5–50 typical)
- `max_iter=300` — number of optimisation steps (more = better but slower)
- **PCA pre-reduction to 10 dims** — standard practice to speed up t-SNE and reduce noise

### KL Divergence (goodness of fit):
t-SNE minimises KL divergence between the high-dimensional and low-dimensional distributions.  
Lower KL = t-SNE did a better job preserving local structure. Typical values: 0.5–3.0

In [ ]:
# ── Prepare t-SNE Sample ──────────────────────────────────────────────────────
tsne_feature_cols = ['Hour', 'Day_Num', 'Month', 'Is_Weekend',
                     'Crime_Severity_Score', 'Latitude', 'Longitude',
                     'District', 'Ward', 'Beat', 'Community Area', 'Arrest', 'Domestic']

tsne_df = df[tsne_feature_cols + ['Primary Type', 'Is_Night']].dropna()
tsne_sample = tsne_df.sample(n=min(ML_SAMPLE, len(tsne_df)), random_state=42).reset_index(drop=True)

scaler_tsne = StandardScaler()
X_tsne_scaled = scaler_tsne.fit_transform(tsne_sample[tsne_feature_cols].values)

print(f't-SNE input: {tsne_sample.shape[0]:,} samples × {len(tsne_feature_cols)} features')
print('  ⏳ Pre-reducing to 10 dims with PCA before t-SNE...')

# Step 1: Pre-reduce to 10 dimensions (standard t-SNE best practice)
# This makes t-SNE faster and removes noisy low-variance dimensions
pca_pre = PCA(n_components=min(10, X_tsne_scaled.shape[1]), random_state=42)
X_pre   = pca_pre.fit_transform(X_tsne_scaled)
print(f'  PCA pre-reduction: {X_tsne_scaled.shape[1]}D → {X_pre.shape[1]}D done!')

In [ ]:
# ── Run t-SNE ─────────────────────────────────────────────────────────────────
# ⏳ WARNING: This takes 2-5 minutes on 15,000 samples
print('Running t-SNE (this takes a few minutes)...')

tsne = TSNE(
    n_components=2,   # Project to 2D for visualisation
    perplexity=30,    # Neighbourhood size (balances local vs global structure)
    max_iter=300,     # Optimisation iterations
    random_state=42   # For reproducibility
)
X_tsne_2d = tsne.fit_transform(X_pre)

print(f'\n✅ t-SNE complete!')
print(f'   Output shape : {X_tsne_2d.shape}')
print(f'   KL Divergence: {tsne.kl_divergence_:.4f}  (lower is better, typical range: 0.5–3.0)')

In [ ]:
# ── Save t-SNE Results + Log to MLflow ───────────────────────────────────────
tsne_result = tsne_sample.copy()
tsne_result['TSNE1'] = X_tsne_2d[:, 0]
tsne_result['TSNE2'] = X_tsne_2d[:, 1]

night_map = {1: 'Night (8PM-6AM)', 0: 'Day (6AM-8PM)'}
tsne_result['Time_Period'] = tsne_result['Is_Night'].astype(int).map(night_map)
tsne_result.to_csv(os.path.join(CLEAN_DIR, 'tsne_result.csv'), index=False)

with mlflow.start_run(run_name='tSNE_Visualization'):
    mlflow.log_params({'perplexity': 30, 'max_iter': 300, 'n_components': 2})
    mlflow.log_metrics({'kl_divergence': round(float(tsne.kl_divergence_), 4)})
    print('✅ t-SNE logged to MLflow')

dim_metrics['tSNE'] = {'kl_divergence': round(float(tsne.kl_divergence_), 4)}

# ── Visualise: Day vs Night Side-by-Side ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, period, color in zip(axes,
                              ['Day (6AM-8PM)', 'Night (8PM-6AM)'],
                              ['#2196F3', '#FF5722']):
    mask = tsne_result['Time_Period'] == period
    ax.scatter(tsne_result.loc[mask, 'TSNE1'],
               tsne_result.loc[mask, 'TSNE2'],
               s=4, alpha=0.4, color=color)
    ax.set_title(f't-SNE — {period}\n(n={mask.sum():,} crimes)', fontweight='bold', fontsize=12)
    ax.set_xlabel('t-SNE Dimension 1')
    ax.set_ylabel('t-SNE Dimension 2')

plt.suptitle(f't-SNE 2D Embedding of Chicago Crimes\nKL Divergence = {tsne.kl_divergence_:.4f}',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Model Comparison & Final Report <a id='9-compare'></a>

Now we compare all algorithms side-by-side and save the final report files.

In [ ]:
# ── Silhouette Score Comparison ───────────────────────────────────────────────
all_metrics = {**geo_metrics, **temporal_metrics}

comparison_rows = []
for name, m in all_metrics.items():
    comparison_rows.append({
        'Algorithm'     : name,
        'Silhouette'    : m.get('silhouette', 0),
        'Davies_Bouldin': m.get('davies_bouldin', 0),
        'N_Clusters'    : m.get('n_clusters', 'N/A')
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(os.path.join(CLEAN_DIR, 'silhouette_scores.csv'), index=False)

print('=== FINAL MODEL COMPARISON ===')
print(comparison_df.to_string(index=False))

# Save dim reduction summary
with open(os.path.join(CLEAN_DIR, 'dimensionality_reduction_summary.json'), 'w') as f:
    json.dump(dim_metrics, f, indent=2)

In [ ]:
# ── Final Comparison Bar Chart ────────────────────────────────────────────────
geo_comp = comparison_df[comparison_df['Algorithm'].str.contains('Geo')].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Silhouette Score (higher = better)
bars = axes[0].bar(geo_comp['Algorithm'], geo_comp['Silhouette'],
                   color=['#4ade80', '#60a5fa', '#f59e0b'], edgecolor='black')
axes[0].axhline(0.5, color='red', linestyle='--', label='Target: 0.5')
axes[0].set_title('Silhouette Score (↑ Higher = Better)', fontweight='bold')
axes[0].set_ylim(0, max(geo_comp['Silhouette']) * 1.2)
axes[0].set_xticklabels(geo_comp['Algorithm'], rotation=15)
axes[0].legend()
for bar, val in zip(bars, geo_comp['Silhouette']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

# Davies-Bouldin (lower = better)
bars2 = axes[1].bar(geo_comp['Algorithm'], geo_comp['Davies_Bouldin'],
                    color=['#4ade80', '#60a5fa', '#f59e0b'], edgecolor='black')
axes[1].set_title('Davies-Bouldin Index (↓ Lower = Better)', fontweight='bold')
axes[1].set_xticklabels(geo_comp['Algorithm'], rotation=15)
for bar, val in zip(bars2, geo_comp['Davies_Bouldin']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Geographic Clustering Algorithm Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Final Combined Sample (for Streamlit App) ─────────────────────────────────
geo_small = geo_sample.sample(n=min(ML_SAMPLE, len(geo_sample)), random_state=42).reset_index(drop=True)
geo_small.to_csv(os.path.join(CLEAN_DIR, 'sample_clusters.csv'), index=False)

print('\n✅ All outputs saved to data/cleaned/')
print('\nFiles generated:')
for f in sorted(os.listdir(CLEAN_DIR)):
    size = os.path.getsize(os.path.join(CLEAN_DIR, f))
    print(f'  {f:<45} {size/1024:>8.1f} KB')

---
## ✅ ML Pipeline Complete!

### Summary of all experiments run:

| Algorithm | Type | Silhouette | Davies-Bouldin | Best? |
|-----------|------|-----------|----------------|-------|
| K-Means (k=8) | Geographic | ~0.41 | ~0.80 | ⚡ Fastest |
| **DBSCAN** | Geographic | **~0.55** | **~0.37** | **✅ Best overall** |
| Hierarchical (k=8) | Geographic | ~0.36 | ~0.81 | 🌳 Best dendrogram |
| K-Means (k=4) | Temporal | ~0.21 | ~1.43 | ⏰ Only temporal |
| PCA | Dim. Reduction | 41.6% (2D) | — | 📊 6 PCs = 70% var |
| t-SNE | Dim. Reduction | KL≈2.77 | — | 🔬 Best for vis |

### 🏆 Recommendation: DBSCAN
> DBSCAN achieves the highest silhouette score and lowest Davies-Bouldin index.  
> It discovers Chicago's natural crime hotspot geography **without requiring a pre-set k**,  
> and automatically identifies isolated incidents as noise.  
> Use K-Means when **equal-sized patrol zones** are operationally required.

### View in MLflow:
```bash
cd /home/ganesaperumal/Documents/DS
source venv/bin/activate
mlflow ui
# Open http://localhost:5000
```

### Launch the Streamlit App:
```bash
streamlit run app.py
# Open http://localhost:8501
```